<p style="text-align:center"> 
    <a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/" target="_blank"> 
    <img src="../assets/logo.png" width="200" alt="Flavio Aguirre Logo"> 
    </a>
</p>

<h1 align="center"><strong>Weather Wise – 04 · Preprocessing and Modeling</strong></h1>
<hr>

In this notebook we:

- Load the **feature–engineered dataset** from the previous step.
- Split the data into **train** and **test** sets with stratification.
- Build a preprocessing pipeline that:
  - scales numerical features,
  - one–hot encodes categorical features.
- Train a **Random Forest classifier** using cross–validated grid search.
- Evaluate the model on a held–out test set.
- Persist the best model to disk for later evaluation and deployment.

This is the core modeling step of the project.

In [1]:
## Preprocessing and Modeling

import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")

We now use the **feature–engineered dataset** produced in the previous notebook.

In [3]:
# Load dataset with engineered features
input_path = "../data/processed/weatherAUS-data-engineered.csv"
df = pd.read_csv(input_path)

print(f"Data with engineered features loaded from {input_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

df.head()

Data with engineered features loaded from ../data/processed/weatherAUS-data-engineered.csv
Shape: 7557 rows x 31 columns


,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,RainToday,Season,TempDiff,TempChange,PressureDiff,HumidityDiff,WindSpeedDiff,AvgHumidity,AvgTemp,RainfallPerSunshine
0,MelbourneAirport,11.2,19.9,0.0,5.6,8.8,SW,69.0,W,SW,...,Yes,Summer,8.7,2.2,1.3,-18.0,10.0,46.0,17.00,0.000000
1,MelbourneAirport,7.8,17.8,1.2,7.2,12.9,SSE,56.0,SW,SSE,...,No,Summer,10.0,3.3,1.3,-7.0,-5.0,46.5,14.15,0.092308
2,MelbourneAirport,6.3,21.1,0.0,6.2,10.5,SSE,31.0,E,S,...,No,Summer,14.8,6.2,-3.2,-16.0,6.0,43.0,16.50,0.000000
3,MelbourneAirport,8.1,29.2,0.0,6.4,12.5,SSE,35.0,NE,SSE,...,No,Summer,21.1,12.2,-3.4,-44.0,18.0,45.0,22.10,0.000000
4,MelbourneAirport,9.7,29.0,0.0,7.4,12.3,SE,33.0,SW,SSE,...,No,Summer,19.3,7.7,-1.6,-20.0,11.0,41.0,23.25,0.000000


## 1. Define features and target

Our prediction target remains **`RainToday`** (at least 1 mm of rain today),  
redefined earlier to avoid data leakage.

We separate the feature matrix (`X`) and target vector (`y`).

In [5]:
# Define the features and target variable
X = df.drop(columns=["RainToday"])
y = df["RainToday"]

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")

Feature matrix X shape: (7557, 30)
Target vector y shape: (7557,)


## 2. Train / test split with stratification

We split the dataset into:

- 80% training set,
- 20% test set,

using **stratification on the target** so that the class proportions are preserved in both splits.

Even though the classes are only **moderately imbalanced** (around 3:1),  
stratification is a good general practice.

In [6]:
# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train shape: {X_train.shape}, {y_train.shape}")
print(f"Test shape: {X_test.shape}, {y_test.shape}")

Train shape: (6045, 30), (6045,)
Test shape: (1512, 30), (1512,)


In [39]:
print(f"Numeric features: {numeric_features}\n")
print(f"Categorical features: {categorical_features}")

Numeric features: ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'TempDiff', 'TempChange', 'PressureDiff', 'HumidityDiff', 'WindSpeedDiff', 'AvgHumidity', 'AvgTemp', 'RainfallPerSunshine']

Categorical features: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainYesterday', 'Season']


## 3. Preprocessing: numeric vs categorical features

We automatically detect numerical and categorical columns and map them to separate preprocessing pipelines:

- **Numerical:** standardized with `StandardScaler`.
- **Categorical:** one–hot encoded with `OneHotEncoder(handle_unknown="ignore")`.

In [7]:
# Detect numeric and categorical features
numeric_features = X_train.select_dtypes(include=["float", "int"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}\n")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

Numeric features (24): ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'TempDiff', 'TempChange', 'PressureDiff', 'HumidityDiff', 'WindSpeedDiff', 'AvgHumidity', 'AvgTemp', 'RainfallPerSunshine']

Categorical features (6): ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainYesterday', 'Season']


In [8]:
# Scale numeric features
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

# One-hot encode categorical features
categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [9]:
# Combine both transformers into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 4. Modeling: Random Forest classifier in a pipeline

We now create a **single pipeline** that chains:

1. The preprocessing step (`preprocessor`),
2. A **Random Forest classifier**.

This ensures that:

- The same preprocessing is consistently applied during cross–validation and on the test set,
- We can easily deploy the full pipeline (preprocessing + model) as a single object.

<details>
    <summary><strong>Why Random Forest?</strong></summary>
    <p>
    A Random Forest classifier is a good starting point for this problem because:
    </p>
    <ul>
        <li>It can capture non-linear relationships between features and the target.</li>
        <li>It handles mixed feature types (numerical + categorical via one-hot encoding).</li>
        <li>It is relatively robust to outliers and collinearity.</li>
        <li>It provides feature importance estimates, which help interpret the model.</li>
    </ul>
    <p>
    This does not mean we cannot experiment with other models (e.g. Logistic Regression, Gradient Boosting) in future iterations, but Random Forest is a strong, interpretable baseline.
    </p>
</details>


In [10]:
# Create the pipeline combining preprocessing and the classifier
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ]
)

## 5. Hyperparameter tuning with cross‑validated grid search

We define a small **parameter grid** for the Random Forest and use `GridSearchCV` with **StratifiedKFold**:

- This searches over a combination of:
  - `n_estimators` (number of trees),
  - `max_depth` (tree depth),
  - `min_samples_split` (minimum samples to split a node).
- Cross‑validation uses the same target stratification logic as the train/test split.

For scoring, we start with **accuracy** to keep the grid compact and fast.  
In a more advanced iteration, we could optimize directly for **F1 (rain class)**.

In [11]:
# Define parameter grid for cross-validated grid search
param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
}

# Stratified K-Fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [12]:
# Instantiate GridSearchCV with the full pipeline
model = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    verbose=2,
    n_jobs=-1
)

# Fit on the training data
model.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['MinTemp',
                                                                          'MaxTemp',
                                                                          'Rainfall',
                                                                          'Evaporation',
                                                                          'Sunshine',
                                                                          'WindGustSpeed',
                                                                          'WindSpeed9am',
                                                                          'WindSpeed3pm',
                                                                          'Humidity9am',
                                                                          'Humidity3pm',
                                                                          'Pressure9am',
                                                                          'Pres...
                                                                         Pipeline(steps=[('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Location',
                                                                          'WindGustDir',
                                                                          'WindDir9am',
                                                                          'WindDir3pm',
                                                                          'RainYesterday',
                                                                          'Season'])])),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [None, 10, 20],
                         'classifier__min_samples_split': [2, 5],
                         'classifier__n_estimators': [50, 100]},
             scoring='accuracy', verbose=2)

## 6. Best parameters and cross‑validation performance

In [13]:
print(f"Best parameters found: {model.best_params_}\n")
print(f"Best cross-validation score (accuracy): {model.best_score_:.3f}")

Best parameters found: {'classifier__max_depth': None, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}

Best cross-validation score (accuracy): 0.855


## 7. Evaluation on the test set

We now evaluate the best model (selected during cross‑validation) on the **held‑out test set**.

This gives us an **unbiased estimate** of how well the model generalizes to new data.

In [14]:
test_score = model.score(X_test, y_test)
print(f"Test set accuracy: {test_score:.3f}")

Test set accuracy: 0.845


With this configuration, we obtain a **reasonably accurate classifier**,  
expected to correctly predict whether it will rain today in the Melbourne area in roughly **85%** of cases.

A deeper evaluation of the model (confusion matrix, F1, ROC–AUC, feature importance)  
will be performed in the next notebook.

## 8. Persist the best model

Finally, we save the fitted `GridSearchCV` object, which includes:

- The best estimator (full preprocessing + Random Forest),
- The cross‑validation results (if needed for future analysis).

This serialized model can later be loaded for:

- Detailed evaluation,
- Integration into an API or Streamlit app,
- Further refinement.

In [16]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, "model_randomforest_precipicheck_26-11-2025.pkl")
joblib.dump(model, model_path)

print(f"Saved trained model (GridSearchCV + pipeline) to: {model_path}")

Saved trained model (GridSearchCV + pipeline) to: ../models\model_randomforest_precipicheck_26-11-2025.pkl


<hr>

## Author

<a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/">**Flavio Aguirre**</a>
<br>
<a href="https://coursera.org/share/e27ae5af81b56f99a2aa85289b7cdd04">***Data Scientist***</a>